# CivicPulse AI — Qwen3-VL free-GPU prototype

Prototype only. Runs interactively in a free Colab GPU session; it is not an always-on API. Do not upload real citizen/private images unless you have authorization and have reviewed the notebook/runtime privacy implications. Use synthetic, public, or consented test images.

Model: `Qwen/Qwen3-VL-4B-Instruct` (Apache-2.0). The notebook uses 4-bit loading to reduce VRAM use. GPU availability and runtime limits are controlled by the notebook provider.

In [ ]:
!pip -q install -U "transformers>=4.57.0" accelerate bitsandbytes qwen-vl-utils pillow

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU, if available.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Model loaded:", MODEL_ID)

## Phase 1 — Batch evaluation on labelled images

Use a small, manually labelled test set first. Upload multiple images in one run. Name each file with its ground-truth label as the first filename segment, followed by `__`, for example `pothole__001.jpg`, `garbage__002.png`, `sewage__003.jpg`, `streetlight__004.jpg`, `none__005.jpg`, or `unclear__006.jpg`.

Allowed ground-truth labels: `pothole`, `garbage`, `sewage`, `streetlight`, `none`, `unclear`. `none` means no civic defect is visible. Use `unclear` only when even a human reviewer cannot reliably determine the scene/category. Keep test images separate from prompt-tuning images. Use public, synthetic, or consented test images—not private citizen reports.

Suggested first batch: 5–10 images per class. Expand toward the 85-image plan after the workflow is stable.

In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload labelled test images first.')

VALID_LABELS = {'pothole', 'garbage', 'sewage', 'streetlight', 'none', 'unclear'}
evaluation_items = []
for filename in uploaded.keys():
    prefix = filename.rsplit('/', 1)[-1].split('__', 1)[0].lower()
    if prefix not in VALID_LABELS:
        print(f'SKIP {filename}: filename must start with one of {sorted(VALID_LABELS)} followed by __')
        continue
    evaluation_items.append({'filename': filename, 'ground_truth': prefix})

if not evaluation_items:
    raise ValueError('No correctly named files found. Example: pothole__001.jpg')
print(f'Queued {len(evaluation_items)} labelled images.')

In [ ]:
import json, re, time, os, pandas as pd
from PIL import Image
from qwen_vl_utils import process_vision_info

TRIAGE_PROMPT = '''You are an image-triage assistant for a civic issue reporting system.
Analyze only visible evidence. Do not infer an exact address, identity, cause, or hidden facts.
Return exactly one JSON object, without markdown, with these keys:
{
  "defect_type": "pothole|garbage|sewage|streetlight|none|unclear",
  "confidence": 0.0,
  "severity": "low|medium|high|unclear",
  "safety_risk": "low|medium|high|unclear",
  "summary": "short description of visible evidence",
  "evidence": ["observable visual detail"],
  "recommended_action": "suggested municipal inspection/action",
  "human_review_required": true
}
Use defect_type=none only when no civic defect is visible. Use unclear when the image is too blurry, dark, obstructed, or ambiguous to classify. Be conservative. Confidence must be a number from 0 to 1. Always set human_review_required=true.'''

def analyze_one_image(image_path):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image_path},
            {'type': 'text', 'text': TRIAGE_PROMPT + '\\nAnalyze this image for civic issue triage.'},
        ],
    }]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(model.device)
    started = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    elapsed = time.perf_counter() - started
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    raw_text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    match = re.search(r'\{.*\}', raw_text, flags=re.S)
    if not match:
        return {'raw_output': raw_text, 'json_valid': False, 'inference_seconds': elapsed}
    try:
        data = json.loads(match.group(0))
        allowed = {'pothole', 'garbage', 'sewage', 'streetlight', 'none', 'unclear'}
        label = str(data.get('defect_type', '')).lower()
        confidence = float(data.get('confidence', 0))
        if label not in allowed or not 0 <= confidence <= 1:
            raise ValueError('Invalid defect_type or confidence value')
        if data.get('human_review_required') is not True:
            raise ValueError('human_review_required must be true')
        return {'prediction': label, 'confidence': confidence, 'severity': data.get('severity'),
                'safety_risk': data.get('safety_risk'), 'summary': data.get('summary'),
                'evidence': json.dumps(data.get('evidence', []), ensure_ascii=False),
                'recommended_action': data.get('recommended_action'), 'json_valid': True,
                'human_review_required': True, 'raw_output': raw_text, 'inference_seconds': elapsed}
    except Exception as exc:
        return {'raw_output': raw_text, 'json_valid': False, 'parse_error': str(exc), 'inference_seconds': elapsed}

print('Evaluation function ready.')

In [ ]:
rows = []
for index, item in enumerate(evaluation_items, start=1):
    print(f"[{index}/{len(evaluation_items)}] {item['filename']}")
    try:
        output = analyze_one_image(item['filename'])
        rows.append({'filename': item['filename'], 'ground_truth': item['ground_truth'], **output})
    except Exception as exc:
        rows.append({'filename': item['filename'], 'ground_truth': item['ground_truth'],
                     'json_valid': False, 'error': str(exc), 'inference_seconds': None})

results_df = pd.DataFrame(rows)
display(results_df[['filename', 'ground_truth'] + [c for c in ['prediction','confidence','json_valid','inference_seconds','error'] if c in results_df.columns]])
csv_path = 'civicpulse_qwen_phase1_predictions.csv'
results_df.to_csv(csv_path, index=False)
print('Saved:', csv_path)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

valid_df = results_df[results_df.get('prediction', pd.Series(index=results_df.index, dtype='object')).notna()].copy()
print('Total images:', len(results_df))
print('Valid JSON responses:', int(results_df['json_valid'].fillna(False).sum()))
print('JSON validity rate:', round(float(results_df['json_valid'].fillna(False).mean()) * 100, 2), '%')
if len(valid_df):
    labels = ['pothole', 'garbage', 'sewage', 'streetlight', 'none', 'unclear']
    print('Classification accuracy:', round(accuracy_score(valid_df['ground_truth'], valid_df['prediction']) * 100, 2), '%')
    print('\\nPer-class precision / recall / F1:')
    print(classification_report(valid_df['ground_truth'], valid_df['prediction'], labels=labels, zero_division=0))
    print('Confusion matrix (rows=true, columns=predicted):')
    display(pd.DataFrame(confusion_matrix(valid_df['ground_truth'], valid_df['prediction'], labels=labels), index=labels, columns=labels))
    if 'inference_seconds' in valid_df:
        print('Mean inference seconds:', round(valid_df['inference_seconds'].dropna().mean(), 2))
else:
    print('No valid predictions to score yet. Inspect raw_output and parse_error columns.')

files.download(csv_path)

## How to interpret Phase 1

- A small test batch is a workflow check, not proof of production accuracy. Increase the dataset and keep it balanced across classes.
- Review every false positive and false negative manually. A false `none` on a real hazard and a false defect on a normal scene both matter.
- Do not tune the prompt on the final held-out test images. Keep separate development and evaluation sets.
- Confidence is model-reported and uncalibrated. Do not use it alone for automatic rejection or priority assignment.
- All outputs must remain subject to human review. The model must not autonomously reject reports or declare repairs complete.